In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# LOAD DATA

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

targets = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in train.columns if col not in targets + ['index']]

X = train[feature_cols].copy()
y = train[targets].copy()
X_test = test[feature_cols].copy()

In [ ]:
# Fill missing values
X = X.fillna(0)
X_test = X_test.fillna(0)

print(f"Original features: {X.shape[1]}")

In [ ]:
# ====================== FEATURE PROCESSING ======================
# Remove low-variance features
selector = VarianceThreshold(threshold=0.01)
X_selected = selector.fit_transform(X)
X_test_selected = selector.transform(X_test)

selected_features = [col for col, keep in zip(X.columns, selector.get_support()) if keep]
X = pd.DataFrame(X_selected, columns=selected_features)
X_test = pd.DataFrame(X_test_selected, columns=selected_features)

print(f"Features after filter: {X.shape[1]}")

# Scaling
scaler = RobustScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# Log transform
y_log = np.log1p(y)

In [ ]:
# ====================== TRAINING FUNCTION ======================
def train_and_predict(target_name, X, y_target, X_test, n_folds=5):
    print(f"\n=== Training for {target_name} ===")

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    test_preds = np.zeros((len(X_test), n_folds))

    params = {
        'n_estimators': 800,
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'eval_metric': 'rmse'
    }

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y_target.iloc[train_idx]
        y_val = y_target.iloc[val_idx]

        model = xgb.XGBRegressor(**params)

        # Compatible with XGBoost 3.x
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        test_preds[:, fold] = model.predict(X_test)

        oof_pred = model.predict(X_val)
        fold_rmse = np.sqrt(np.mean((y_val.values - oof_pred)**2))
        print(f"Fold {fold} RMSE: {fold_rmse:.4f}")

    final_pred = np.mean(test_preds, axis=1)
    return final_pred


In [ ]:
# ====================== TRAIN MODELS ======================
predictions = {}

for target in targets:
    pred_log = train_and_predict(target, X, y_log[target], X_test)
    predictions[target] = np.expm1(pred_log)

# ====================== CREATE SUBMISSION ======================
submission = pd.DataFrame({
    'index': test['index'],
    'IC50': predictions['IC50, mM'],
    'CC50': predictions['CC50, mM'],
    'SI': predictions['SI']
})

# Post-processing
submission[['IC50', 'CC50', 'SI']] = submission[['IC50', 'CC50', 'SI']].clip(lower=0.0001)

submission.to_csv('submission_improved.csv', index=False)

print(submission.head())